# HW01-A — Dockerized Airbnb Ops Package

Your first job is to make a tiny data package that can run the same way twice.

That sounds boring. Boring reproducibility is how production works.

By the end, you should have:

- a Python package
- a CLI command
- a Docker image
- a Docker Compose service
- a DVC stage
- a processed CSV
- a short run report

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials (for homeworks B & C, not this one). Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

Use docs when stuck. Guessing is slower.

- Python `pyproject.toml`: https://packaging.python.org/en/latest/guides/writing-pyproject-toml/
- Python CLI entry points: https://setuptools.pypa.io/en/latest/userguide/entry_point.html
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker Compose: https://docs.docker.com/compose/
- Docker volumes: https://docs.docker.com/engine/storage/volumes/
- DVC `dvc.yaml`: https://doc.dvc.org/user-guide/project-structure/dvcyaml-files
- DVC `repro`: https://doc.dvc.org/command-reference/repro

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

In [1]:
from pathlib import Path
import textwrap
import pandas as pd

PROJECT = Path.cwd()
for path in ['src/airbnb_ops', 'data/raw', 'data/processed', 'reports', 'tests']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_ops/__init__.py').write_text('__version__ = "0.1.0"\n')
print(PROJECT)

d:\MLOps-Bootcamp-2026\exercises\HW01_A


## 1. Raw inputs

The two cells below are given to you.

Do not change these rows. The point of HW01-A is packaging, Docker, DVC, and validation, not inventing toy data.

Files created:

```text
data/raw/listings_sample.csv
data/raw/neighbourhood_segments.csv
```

The listings file includes `host_name` and `host_id` on purpose. Your PII code must handle them.


In [2]:
# Provided starter data 1.1
# Do not edit this cell.
# Implementation note:
# - This creates the listing-level raw input.
# - The host_name and host_id columns are intentionally included.
# - Later, your PII code must drop host_name and replace host_id with a stable host_key.

listings = pd.DataFrame(
    [
        {
            "listing_id": 1,
            "neighbourhood": "Centrum-West",
            "price": 180,
            "minimum_nights": 2,
            "availability_365": 120,
            "number_of_reviews": 31,
            "host_id": 901,
            "host_name": "Alice",
        },
        {
            "listing_id": 2,
            "neighbourhood": "Centrum-West",
            "price": 210,
            "minimum_nights": 3,
            "availability_365": 80,
            "number_of_reviews": 12,
            "host_id": 902,
            "host_name": "Bob",
        },
        {
            "listing_id": 3,
            "neighbourhood": "De Pijp",
            "price": 135,
            "minimum_nights": 2,
            "availability_365": 210,
            "number_of_reviews": 44,
            "host_id": 903,
            "host_name": "Chris",
        },
        {
            "listing_id": 4,
            "neighbourhood": "Oud-Noord",
            "price": 95,
            "minimum_nights": 1,
            "availability_365": 300,
            "number_of_reviews": 9,
            "host_id": 904,
            "host_name": "Dana",
        },
        {
            "listing_id": 5,
            "neighbourhood": "Oud-Noord",
            "price": 105,
            "minimum_nights": 2,
            "availability_365": 260,
            "number_of_reviews": 18,
            "host_id": 905,
            "host_name": "Eve",
        },
        {
            "listing_id": 6,
            "neighbourhood": "De Baarsjes",
            "price": 125,
            "minimum_nights": 3,
            "availability_365": 170,
            "number_of_reviews": 27,
            "host_id": 906,
            "host_name": "Farid",
        },
        {
            "listing_id": 7,
            "neighbourhood": "De Baarsjes",
            "price": 145,
            "minimum_nights": 4,
            "availability_365": 90,
            "number_of_reviews": 21,
            "host_id": 907,
            "host_name": "Grace",
        },
        {
            "listing_id": 8,
            "neighbourhood": "Westerpark",
            "price": 155,
            "minimum_nights": 2,
            "availability_365": 140,
            "number_of_reviews": 36,
            "host_id": 908,
            "host_name": "Hamed",
        },
    ]
)

listings.to_csv(PROJECT / "data/raw/listings_sample.csv", index=False)
listings.head()


,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_id,host_name
0,1,Centrum-West,180,2,120,31,901,Alice
1,2,Centrum-West,210,3,80,12,902,Bob
2,3,De Pijp,135,2,210,44,903,Chris
3,4,Oud-Noord,95,1,300,9,904,Dana
4,5,Oud-Noord,105,2,260,18,905,Eve


In [3]:
# Provided starter data 1.2
# Do not edit this cell.
# Implementation note:
# - This file enriches neighbourhoods with business metadata.
# - Your transform step should join this file after aggregation.
# - If a neighbourhood has no segment row, your code should fill it with "unknown".

segments = pd.DataFrame(
    [
        {
            "neighbourhood": "Centrum-West",
            "tourism_segment": "tourist-heavy",
            "priority_level": "high",
        },
        {
            "neighbourhood": "De Pijp",
            "tourism_segment": "mixed",
            "priority_level": "medium",
        },
        {
            "neighbourhood": "Oud-Noord",
            "tourism_segment": "emerging",
            "priority_level": "medium",
        },
        {
            "neighbourhood": "De Baarsjes",
            "tourism_segment": "local-heavy",
            "priority_level": "low",
        },
        {
            "neighbourhood": "Westerpark",
            "tourism_segment": "mixed",
            "priority_level": "medium",
        },
    ]
)

segments.to_csv(PROJECT / "data/raw/neighbourhood_segments.csv", index=False)
segments


,neighbourhood,tourism_segment,priority_level
0,Centrum-West,tourist-heavy,high
1,De Pijp,mixed,medium
2,Oud-Noord,emerging,medium
3,De Baarsjes,local-heavy,low
4,Westerpark,mixed,medium


In [4]:
# Checkpoint
for file in ['data/raw/listings_sample.csv', 'data/raw/neighbourhood_segments.csv']:
    assert Path(file).exists(), f'Missing {file}'

pd.read_csv('data/raw/listings_sample.csv').head()

,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_id,host_name
0,1,Centrum-West,180,2,120,31,901,Alice
1,2,Centrum-West,210,3,80,12,902,Bob
2,3,De Pijp,135,2,210,44,903,Chris
3,4,Oud-Noord,95,1,300,9,904,Dana
4,5,Oud-Noord,105,2,260,18,905,Eve


## 2. Data contract

Write code against a contract, not vibes.

Input listing columns:

```text
listing_id, neighbourhood, price, minimum_nights,
availability_365, number_of_reviews, host_id, host_name
```

Output columns:

```text
neighbourhood, num_listings, avg_price, median_price,
avg_minimum_nights, availability_365_avg, total_reviews,
reviews_per_listing, tourism_segment, priority_level
```

In [5]:
# TODO 2.1
# Create src/airbnb_ops/config.py.
# Add a PipelineConfig dataclass with default paths for:
# listings_path, segments_path, output_path, report_path

from pathlib import Path

config_py = """
from dataclasses import dataclass
from pathlib import Path

BASE_DIR = Path.cwd()

@dataclass
class PipelineConfig:
    listings_path: Path = BASE_DIR / "data/raw/listings_sample.csv"
    segments_path: Path = BASE_DIR / "data/raw/neighbourhood_segments.csv"
    output_path: Path = BASE_DIR / "data/processed/airbnb_neighbourhood_summary.csv"
    report_path: Path = BASE_DIR / "reports/hw01_a_run_report.md"
"""

Path("src/airbnb_ops").mkdir(parents=True, exist_ok=True)
Path("src/airbnb_ops/config.py").write_text(config_py)

411

In [6]:
# TODO 2.2
# Create src/airbnb_ops/extract.py.
# Add read_csv_checked(path: Path) -> pd.DataFrame.
# It should raise FileNotFoundError if the file is missing.
extract_py= """
from pathlib import Path
import pandas as pd

def read_csv_checked(path: Path) -> pd.DataFrame:
    '''
    Read a CSV file and return it as a DataFrame.

    Raises:
        FileNotFoundError: If the specified file does not exist.
    '''
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    return pd.read_csv(path)
"""
Path("src/airbnb_ops").mkdir(parents=True, exist_ok=True)
Path("src/airbnb_ops/extract.py").write_text(extract_py)

357

## 3. PII handling

For this homework:

- drop `host_name`
- convert `host_id` to `host_key`
- drop the original `host_id`

Do not use Python's built-in `hash()`. It is not stable across sessions. Use `hashlib.sha256`.

In [7]:
# TODO 3.1
# Create src/airbnb_ops/pii.py.
# Add:
# - DIRECT_PII_COLUMNS
# - pseudonymize_value(value, salt='qbc12')
# - handle_pii(df)
pii_py ="""
from hashlib import sha256
import pandas as pd


DIRECT_PII_COLUMNS = ["host_name"]

def pseudonymize_value(value, salt: str = "qbc12") -> str:
    '''
    Convert a sensitive identifier into a deterministic pseudonymous key.

    Args:
        value: Original identifier value.
        salt: Additional secret string used during hashing.

    Returns:
        SHA-256 hash represented as a hexadecimal string.
    '''

    raw_value = f"{salt}{value}"

    return sha256(raw_value.encode("utf-8")).hexdigest()


def handle_pii(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Remove direct PII and pseudonymize host identifiers.

    Transformations:
    - Drop 'host_name'
    - Convert 'host_id' -> 'host_key'
    - Drop original 'host_id'
    '''

    # Work on a copy to avoid modifying the caller's DataFrame.
    df = df.copy()

    # Remove direct PII columns if they exist.
    existing_pii = [col for col in DIRECT_PII_COLUMNS if col in df.columns]
    df = df.drop(columns=existing_pii)

    # Create a pseudonymous host key from the original host_id.
    if "host_id" in df.columns:
        df["host_key"] = df["host_id"].apply(pseudonymize_value)

        # Remove the original identifier after pseudonymization.
        df = df.drop(columns=["host_id"])

    return df
"""
Path("src/airbnb_ops").mkdir(parents=True, exist_ok=True)
Path("src/airbnb_ops/pii.py").write_text(pii_py)

1282

## 4. Transform

Build a neighbourhood-level summary. Use `groupby`, not loops.

Join the segment metadata after aggregation. If a segment is missing, fill it with `unknown`.

In [8]:
# TODO 4.1
# Create src/airbnb_ops/transform.py.
# Add build_neighbourhood_summary(listings, segments).
# It should validate required input columns, aggregate, join segments, and return a dataframe.

transform_py = """
import pandas as pd

REQUIRED_LISTING_COLUMNS = {
    "neighbourhood",
    "price",
    "minimum_nights",
    "availability_365",
    "number_of_reviews",
}

REQUIRED_SEGMENT_COLUMNS = {
    "neighbourhood",
    "tourism_segment",
    "priority_level",
}


def build_neighbourhood_summary(
    listings: pd.DataFrame,
    segments: pd.DataFrame,
) -> pd.DataFrame:

    missing_listing_cols = REQUIRED_LISTING_COLUMNS - set(listings.columns)
    if missing_listing_cols:
        raise ValueError(
            f"Listings missing required columns: {missing_listing_cols}"
        )

    missing_segment_cols = REQUIRED_SEGMENT_COLUMNS - set(segments.columns)
    if missing_segment_cols:
        raise ValueError(
            f"Segments missing required columns: {missing_segment_cols}"
        )

    summary = (
        listings.groupby("neighbourhood", as_index=False)
        .agg(
            num_listings=("listing_id", "count"),
            avg_price=("price", "mean"),
            median_price=("price", "median"),
            avg_minimum_nights=("minimum_nights", "mean"),
            availability_365_avg=("availability_365", "mean"),
            total_reviews=("number_of_reviews", "sum"),
        )
    )

    summary["reviews_per_listing"] = (
        summary["total_reviews"] / summary["num_listings"]
    )

    summary = summary.merge(
        segments,
        on="neighbourhood",
        how="left",
    )

    summary["tourism_segment"] = summary["tourism_segment"].fillna("unknown")
    summary["priority_level"] = summary["priority_level"].fillna("unknown")

    return summary
"""

Path("src/airbnb_ops").mkdir(parents=True, exist_ok=True)
Path("src/airbnb_ops/transform.py").write_text(transform_py)

1598

## 5. Validation

Validation is the bouncer. Bad output does not get into the club.

Minimum checks:

- output is not empty
- required output columns exist
- no PII columns exist
- neighbourhood is not null
- num_listings > 0
- avg_price >= 0
- availability_365_avg between 0 and 365

In [9]:
# TODO 5.1
# Create src/airbnb_ops/validate.py.
# Add validate_summary(summary) -> None.
# Raise ValueError when a check fails.

validate_py = """
import pandas as pd

# Columns expected in the final neighbourhood summary.
REQUIRED_OUTPUT_COLUMNS = {
    "neighbourhood",
    "num_listings",
    "avg_price",
    "availability_365_avg",
}


# PII columns that must never appear in the final output.
FORBIDDEN_PII_COLUMNS = {
    "host_name",
    "host_id",
}


def validate_summary(summary: pd.DataFrame) -> None:
    '''
    Validate the final neighbourhood summary.

    Raises:
        ValueError: If any validation check fails.
    '''

    # The pipeline should never produce an empty output.
    if summary.empty:
        raise ValueError("Summary output is empty.")

    # Ensure all required columns are present.
    missing_columns = REQUIRED_OUTPUT_COLUMNS - set(summary.columns)
    if missing_columns:
        raise ValueError(
            f"Missing required output columns: {missing_columns}"
        )

    # Ensure no direct PII columns leaked into the output.
    present_pii = FORBIDDEN_PII_COLUMNS.intersection(summary.columns)
    if present_pii:
        raise ValueError(
            f"PII columns found in output: {present_pii}"
        )

    # Every row must belong to a neighbourhood.
    if summary["neighbourhood"].isna().any():
        raise ValueError("Null values found in neighbourhood column.")

    # Aggregated listing counts must be positive.
    if (summary["num_listings"] <= 0).any():
        raise ValueError(
            "num_listings must be greater than 0."
        )

    # Average prices cannot be negative.
    if (summary["avg_price"] < 0).any():
        raise ValueError(
            "avg_price must be non-negative."
        )

    # Average availability must be within valid Airbnb bounds.
    invalid_availability = (
        (summary["availability_365_avg"] < 0)
        | (summary["availability_365_avg"] > 365)
    )

    if invalid_availability.any():
        raise ValueError(
            "availability_365_avg must be between 0 and 365."
        )
"""
Path("src/airbnb_ops").mkdir(parents=True, exist_ok=True)
Path("src/airbnb_ops/validate.py").write_text(validate_py)

1957

## 6. CLI

The package should expose one command:

```bash
airbnb-ops run
```

The command should read inputs, handle PII, transform, validate, write CSV, and write a markdown report.

In [10]:
import sys
from pathlib import Path

src_path = str(Path.cwd() / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(sys.path[0])

d:\MLOps-Bootcamp-2026\exercises\HW01_A\src


In [20]:
# TODO 6.1
# Create src/airbnb_ops/cli.py using Typer.
# Add a run command.
# It should write:
# - data/processed/airbnb_neighbourhood_summary.csv
# - reports/hw01_a_run_report.md

cli_py = """
from pathlib import Path
import typer

from airbnb_ops.config import PipelineConfig
from airbnb_ops.extract import read_csv_checked
from airbnb_ops.pii import handle_pii
from airbnb_ops.transform import build_neighbourhood_summary
from airbnb_ops.validate import validate_summary


app = typer.Typer(
    help="Airbnb neighbourhood summary pipeline."
)


@app.command("run")
def run() -> None:
    '''
    Execute the complete data pipeline.

    Steps:
    1. Read raw datasets.
    2. Remove or pseudonymize PII.
    3. Build neighbourhood-level aggregates.
    4. Validate output quality.
    5. Write output CSV and markdown report.
    '''

    config = PipelineConfig()

    # Load source datasets.
    listings = read_csv_checked(config.listings_path)
    segments = read_csv_checked(config.segments_path)

    # Remove direct PII and pseudonymize identifiers.
    listings = handle_pii(listings)

    # Create neighbourhood-level summary.
    summary = build_neighbourhood_summary(
        listings=listings,
        segments=segments,
    )

    # Ensure output satisfies all quality checks.
    validate_summary(summary)

    # Required homework output locations.
    output_csv = Path(
        "data/processed/airbnb_neighbourhood_summary.csv"
    )

    report_md = Path(
        "reports/hw01_a_run_report.md"
    )

    # Create parent directories if needed.
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    report_md.parent.mkdir(parents=True, exist_ok=True)

    # Persist final dataset.
    summary.to_csv(output_csv, index=False)

    # Generate a simple run report.
    report_contents = f''' Airbnb Pipeline Run Report

## Run Summary

- Rows produced: {len(summary)}
- Columns produced: {len(summary.columns)}

## Output Files

- CSV: {output_csv}
- Report: {report_md}

## Validation

All validation checks passed successfully.
'''

    report_md.write_text(report_contents)

    typer.echo("Pipeline completed successfully.")
    typer.echo(f"Output written to: {output_csv}")
    typer.echo(f"Report written to: {report_md}")

if __name__ == "__main__":
    app()
"""
Path("src/airbnb_ops").mkdir(parents=True, exist_ok=True)
Path("src/airbnb_ops/cli.py").write_text(cli_py)

2101

## 7. Package metadata

`pyproject.toml` makes the project installable. `[project.scripts]` creates the `airbnb-ops` command.

In [21]:
# TODO 7.1
# Create pyproject.toml.
# Requirements:
# - project name: airbnb-ops
# - dependencies: pandas, typer, rich
# - script: airbnb-ops = airbnb_ops.cli:app

from pathlib import Path

pyproject_content = """
[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[project]
name = "airbnb-ops"
version = "0.1.0"
dependencies = [
    "pandas",
    "typer",
    "rich"
]

[project.scripts]
airbnb-ops = "airbnb_ops.cli:app"

[tool.setuptools]
package-dir = {"" = "src"}

[tool.setuptools.packages.find]
where = ["src"]
"""

Path("pyproject.toml").write_text(pyproject_content.strip())

print("Created pyproject.toml")

Created pyproject.toml


In [22]:
# TODO 7.2
# Create requirements.txt with pandas, typer, rich, dvc.

from pathlib import Path

requirements_content = """
pandas
typer
rich
dvc
"""

Path("requirements.txt").write_text(requirements_content.strip())

print("Created requirements.txt")

Created requirements.txt


In [23]:
from airbnb_ops.cli import run

run()

Pipeline completed successfully.
Output written to: data\processed\airbnb_neighbourhood_summary.csv
Report written to: reports\hw01_a_run_report.md


In [24]:
# Local smoke test
import sys
!{sys.executable} -m pip install -q -e .
!airbnb-ops
pd.read_csv('data/processed/airbnb_neighbourhood_summary.csv').head()

Pipeline completed successfully.
Output written to: data\processed\airbnb_neighbourhood_summary.csv
Report written to: reports\hw01_a_run_report.md


,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,availability_365_avg,total_reviews,reviews_per_listing,tourism_segment,priority_level
0,Centrum-West,2,195.0,195.0,2.5,100.0,43,21.5,tourist-heavy,high
1,De Baarsjes,2,135.0,135.0,3.5,130.0,48,24.0,local-heavy,low
2,De Pijp,1,135.0,135.0,2.0,210.0,44,44.0,mixed,medium
3,Oud-Noord,2,100.0,100.0,1.5,280.0,27,13.5,emerging,medium
4,Westerpark,1,155.0,155.0,2.0,140.0,36,36.0,mixed,medium


## 8. Docker

Create `.dockerignore`, `Dockerfile`, and `docker-compose.yml`.

Watch out for this common garbage move: copying generated outputs into the image. Do not do that. Mount `data/` and `reports/` so the container writes outputs to your working directory.

In [25]:
# TODO 8.1
# Create .dockerignore.
# Exclude .git, .venv, __pycache__, .ipynb_checkpoints, data/processed, reports.

from pathlib import Path

dockerignore_content = """
.git
.venv
__pycache__
.ipynb_checkpoints
data/processed
reports
"""

Path(".dockerignore").write_text(dockerignore_content.strip())

print("Created .dockerignore")

Created .dockerignore


In [26]:
# TODO 8.2
# Create Dockerfile.
# Requirements:
# - FROM python:3.11-slim
# - WORKDIR /app
# - copy package metadata and src/
# - install requirements and package
# - default command: airbnb-ops run

from pathlib import Path

dockerfile_content = """
FROM python:3.11-slim

WORKDIR /app

COPY pyproject.toml .
COPY requirements.txt .
COPY src ./src

RUN pip install --no-cache-dir -r requirements.txt
RUN pip install --no-cache-dir .

CMD ["airbnb-ops", "run"]
"""

Path("Dockerfile").write_text(dockerfile_content.strip())

print("Created Dockerfile")

Created Dockerfile


In [27]:
# TODO 8.3
# Create docker-compose.yml.
# Requirements:
# - service name: airbnb-ops
# - build current directory
# - mount ./data and ./reports
# - command: airbnb-ops run

from pathlib import Path

docker_compose_content = """
services:
  airbnb-ops:
    build: .
    volumes:
      - ./data:/app/data
      - ./reports:/app/reports
    command: airbnb-ops run
"""

Path("docker-compose.yml").write_text(docker_compose_content.strip())

print("Created docker-compose.yml")

Created docker-compose.yml


In [ ]:
# Docker smoke test. Run in terminal if notebook cannot access Docker.
!docker compose build
!docker compose run --rm airbnb-ops

## 9. DVC

A DVC stage is the receipt for how your output was produced.

Create one stage named `run_pipeline`.

In [29]:
# TODO 9.1
# Create dvc.yaml.
# Stage:
# - cmd: airbnb-ops run
# - deps: src/airbnb_ops, both raw CSV files
# - outs: processed CSV
# - metrics/report: reports/hw01_a_run_report.md with cache: false

from pathlib import Path

dvc_yaml_content = """
stages:
  pipeline:
    cmd: airbnb-ops
    deps:
      - src/airbnb_ops
      - data/raw/listings_sample.csv
      - data/raw/neighbourhood_segments.csv
    outs:
      - data/processed/airbnb_neighbourhood_summary.csv
    metrics:
      - reports/hw01_a_run_report.md:
          cache: false
"""

Path("dvc.yaml").write_text(dvc_yaml_content.strip())
print("Created dvc.yaml")

Created dvc.yaml


In [ ]:
# Extra credit 9.2
# Run these commands if DVC is installed in your local environment.
# This is a useful check, but the required part is creating a correct dvc.yaml file.

# !dvc repro
# !dvc dag


## Final proof

If this cell fails, HW01-A is not done.

In [30]:
output = Path('data/processed/airbnb_neighbourhood_summary.csv')
report = Path('reports/hw01_a_run_report.md')
assert output.exists(), 'Output file was not created.'
assert report.exists(), 'Report file was not created.'

df = pd.read_csv(output)
required = {'neighbourhood','num_listings','avg_price','median_price','avg_minimum_nights','availability_365_avg','total_reviews','reviews_per_listing','tourism_segment','priority_level'}
assert not (required - set(df.columns)), f'Missing: {required - set(df.columns)}'
for bad in ['host_name','host_id','reviewer_name','reviewer_id','listing_url','host_url']:
    assert bad not in df.columns, f'PII leaked: {bad}'
df.head()

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,availability_365_avg,total_reviews,reviews_per_listing,tourism_segment,priority_level
0,Centrum-West,2,195.0,195.0,2.5,100.0,43,21.5,tourist-heavy,high
1,De Baarsjes,2,135.0,135.0,3.5,130.0,48,24.0,local-heavy,low
2,De Pijp,1,135.0,135.0,2.0,210.0,44,44.0,mixed,medium
3,Oud-Noord,2,100.0,100.0,1.5,280.0,27,13.5,emerging,medium
4,Westerpark,1,155.0,155.0,2.0,140.0,36,36.0,mixed,medium


## Commit

```bash
git add .
git commit -m "HW01-A dockerized Airbnb package"
```